# ClinProtGym ESM-C SAE Analysis

This notebook summarizes the ClinProtGym ESM-C / DeltaEmbSAE pipeline outputs. The heavy work is run by `scripts/clinprotgym_esmc_sae_pipeline.py`; this notebook loads the generated tables and figures for inspection.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "scripts" / "clinprotgym_esmc_sae_pipeline.py").is_file():
    for parent in REPO_ROOT.parents:
        if (parent / "scripts" / "clinprotgym_esmc_sae_pipeline.py").is_file():
            REPO_ROOT = parent
            break

sys.path.insert(0, str(REPO_ROOT))

OUTPUT_ROOT = REPO_ROOT / "data" / "clinprotgym_esmc_sae"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"

print(f"Repo root: {REPO_ROOT}")
print(f"ClinProtGym output root: {OUTPUT_ROOT}")

## Dataset Preparation Summary

In [ ]:
manifest_path = TABLE_DIR / "clinprotgym_processing_manifest.csv"
if not manifest_path.is_file():
    raise FileNotFoundError(f"Missing processing manifest: {manifest_path}. Run the pipeline prepare step first.")

manifest_df = pd.read_csv(manifest_path)
display_cols = [
    "dataset",
    "kept_rows",
    "reference_length",
    "trajectory_replicates",
    "trajectory_columns",
    "functional_score_rows",
    "binary_clinvar_rows",
]
display(manifest_df[[col for col in display_cols if col in manifest_df.columns]])

## Benchmark Summary

In [ ]:
summary_path = TABLE_DIR / "clinprotgym_average_spearman_auc_summary.csv"
best_rows_path = TABLE_DIR / "clinprotgym_best_method_rows_by_dataset.csv"

if summary_path.is_file():
    summary_df = pd.read_csv(summary_path)
    display(summary_df)
else:
    print(f"No average summary yet: {summary_path}")

if best_rows_path.is_file():
    best_rows_df = pd.read_csv(best_rows_path)
    display_cols = [
        "dataset",
        "method_family",
        "model_short",
        "layer",
        "spearman_rho",
        "spearman_target",
        "auc",
        "n_score_sequences",
        "n_variants",
    ]
    display(best_rows_df[[col for col in display_cols if col in best_rows_df.columns]])
else:
    print(f"No best-row summary yet: {best_rows_path}")

## Cross-Replicate Consistency

The final cell writes and displays the dataset-level cross-replicate consistency plot. Consistency is the mean pairwise Pearson correlation across replicate-specific enrichment ratios or selection coefficients from completed benchmark/ensemble artifacts. This cell does not run popDMS, raw-embedding, SAE, or ensemble inference itself; those dots appear only after their real pipeline tasks have been run and collected. Datasets are sorted by their best available method; datasets without at least two real trajectory replicates are retained on the x-axis without dots.

In [ ]:
from scripts.clinprotgym_esmc_sae_pipeline import write_cross_replicate_consistency_outputs

consistency_df, plot_df, consistency_table_path, consistency_figure_path = write_cross_replicate_consistency_outputs(
    OUTPUT_ROOT,
    include_all_datasets=True,
)

if plot_df.empty:
    print("No finite cross-replicate consistency rows are available yet.")
else:
    display(Image(filename=str(consistency_figure_path)))
    display(
        plot_df.sort_values(["dataset", "cross_replicate_consistency"], ascending=[True, False])[
            [
                "dataset",
                "method_label",
                "cross_replicate_consistency",
                "n_replicate_pairs",
                "selection_gamma",
                "model_short",
                "layer",
            ]
        ]
    )

print(f"Cross-replicate consistency table: {consistency_table_path}")
print(f"Cross-replicate consistency figure: {consistency_figure_path}")